Environment

In [ ]:
import sys

libs = {
    "hypernet": "/home/zanardi/Codes/TC/HyperNet/hypernet"
}
for lib in libs.values():
    sys.path.append(lib)

Import libraries

In [ ]:
import os
import numpy as np
import scipy as sp
import joblib as jl
import dask.array as da

from collections import ChainMap
from silx.io.dictdump import h5todict

from functions import *
from hypernet import const

Define inputs

In [ ]:
atom = "O"
nb_cases = 400
nb_workers = 16
# Plotting
saving = True
showing = False

Define paths

In [ ]:
paths = {
    "data": "./../data/",
    "figs": "./figs/"
}
paths["sols"] = paths["data"] + "/sols/"
for k in ("figs",):
    os.makedirs(paths[k], exist_ok=True)

Read solutions

In [ ]:
def read_traj(case_nb, path):
    case_id = str(int(case_nb+1)).zfill(6)
    return {case_id: h5todict(path + f"/{case_id}.hdf5")}

In [ ]:
sols = jl.Parallel(nb_workers)(
    jl.delayed(read_traj)(
        case_nb=i, path=paths["sols"]
    ) for i in range(nb_cases)
)
sols = dict(ChainMap(*sols))

Plot solutions

In [ ]:
def get_xy(sol, pscale=False, minmax=False):
    x = sol["t"]
    y = sol[f"X_{atom}"] * sol["n"]
    if pscale:
        x = x * y * const.UKB * sol["T"]
    if minmax:
        y = (y - np.amin(y)) / (np.amax(y) - np.amin(y))
    return x.squeeze(), y.squeeze()

> Function of $t$

In [ ]:
plot_traj(
    xy=[get_xy(sol, pscale=False, minmax=True) for sol in sols.values()],
    labels=['$t$ [s]', '$\\tilde{n}_{o}$'],
    scales=['log', 'linear'],
    figname=paths["figs"] + "/no_data.png",
    save=saving,
    show=showing
)

> Function of $t\cdot p_\text{o}$

In [ ]:
plot_traj(
    xy=[get_xy(sol, pscale=True, minmax=True) for sol in sols.values()],
    labels=[r'$t\cdot p_{o}(t)$ [s Pa]', '$\\tilde{n}_{o}$'],
    scales=['log', 'linear'],
    figname=paths["figs"] + "/no_data_pscale.png",
    save=saving,
    show=showing
)

Perfoming SVD
> Interpolate solutions onto the same grid

In [ ]:
tp_lim = (-7, 0)
tp = np.logspace(*tp_lim,1000)

In [ ]:
y = []
for sol in sols.values():
    xy = get_xy(sol, pscale=True, minmax=False)
    yp = sp.interpolate.interp1d(*xy, kind='cubic')(tp)
    y.append(yp)
y = np.vstack(y).T

> SVD decomposition

In [ ]:
yn = y / const.UNA
m = np.mean(yn, axis=0, keepdims=True)
s = np.std(yn, axis=0, keepdims=True)
yn = (yn - m) / s

In [ ]:
yd = da.from_array(yn, chunks="auto")
u, s, vh = da.linalg.svd_compressed(yd, k=200)
s = s.compute()
u = u.compute()

> Singular values decay

In [ ]:
cs = 1.0 / np.sum(s**2)
cs *= np.cumsum(s**2)

In [ ]:
# Number of principal components
eps = 1e-5
nb_pc = np.where(cs > 1-eps)[0][0] + 1
nb_pc

In [ ]:
plot_cumenergy(
  cs[:10],
  figname=paths["figs"] + "/cum_en.png",
  save=saving,
  show=showing
)

> Extract modes

In [ ]:
path = "./tensor/"
os.makedirs(path, exist_ok=True)
phi = u[:,:nb_pc]
np.savetxt(path + "/phi.txt", phi, fmt='%.10e')
np.savetxt(path + "/tp.txt", tp, fmt='%.10e')

In [ ]:
path = paths["figs"] + "/modes/"
os.makedirs(path, exist_ok=True)

In [ ]:
path = paths["figs"] + "/modes/"
os.makedirs(path, exist_ok=True)
for i in range(nb_pc):
    mode_nb = str(i+1)
    plot_2d(
        x=tp,
        y_true=phi[:,i],
        labels=[r'$t\cdot p_{o}(t)$ [s Pa]', r"$\phi_{{%s}}$" % mode_nb],
        scales=["log", "linear"],
        filename=path + f"/phi_{mode_nb.zfill(2)}.png",
        save=saving,
        show=showing
    )

> Test reconstruction

In [ ]:
path = paths["figs"] + "/rec/"
os.makedirs(path, exist_ok=True)

In [ ]:
def interpolate(t, sol):
    xy = get_xy(sol, pscale=True, minmax=False)
    return sp.interpolate.interp1d(*xy, kind='cubic')(t)

def reconstruct(y):
    y = y / const.UNA
    m = np.mean(y)
    s = np.std(y)
    yn = (y - m) / s
    w = ols(phi, yn)
    yr = (phi @ w) * s + m
    return yr * const.UNA

In [ ]:
cases = np.random.randint(low=0, high=nb_cases, size=10)

In [ ]:
for icase in cases:
    icase = str(icase).zfill(6)
    yi = interpolate(tp, sols[icase])
    yr = reconstruct(yi)
    plot_2d(
        x=tp,
        y_true=yi,
        y_pred=yr,
        labels=[r'$t\cdot p_{o}(t)$ [s Pa]', r"$n_{o}$ [m$^{-3}$]"],
        scales=["log", "linear"],
        filename=path + f"/{icase}.png",
        save=saving,
        show=showing
    )